## Offline submission notebook
This version has been stripped of everything that needs internet access (package installs, Hugging Face Hub login/push, MLflow/DagsHub tracking) so it can run with Kaggle's internet toggle **off**.

**Before running:** attach the `facebook/mbart-large-50-many-to-many-mmt` model as a Kaggle *Dataset* input (search it in Kaggle Models/Datasets, or upload the files yourself), then set `MODEL_PATH` below to that input's directory. With internet off, `from_pretrained("facebook/...")` cannot download the model, so it must be loaded from local files.

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict

# 1. Load the data
df = pd.read_csv("/kaggle/input/competitions/deep-past-initiative-machine-translation/train.csv")

# Drop rows with missing values in the clean columns
# df = df.dropna(subset=["transliteration_clean", "translation_clean"]).reset_index(drop=True)
df = df.dropna(subset=["transliteration", "translation"]).reset_index(drop=True)

# 2. Split into train and validation (90/10)
train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)

# Convert to Hugging Face Dataset
dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "validation": Dataset.from_pandas(val_df)
})


In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, set_seed

# Set global random seed for reproducibility
set_seed(42)

# Point this at a local/attached-dataset path (no internet download).
# e.g. "/kaggle/input/mbart-large-50-many-to-many-mmt" if you attached the model as a Kaggle dataset.
MODEL_PATH = "/kaggle/input/models/nensipansuriya1311/facebookmbart-large-50-many-to-many-mmt/pytorch/default/1"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

# Set language tokens: Akkadian isn't in standard mBART's vocabulary,
# so we reuse an existing language code for both source and target
tokenizer.src_lang = "en_XX"
tokenizer.tgt_lang = "en_XX"

MAX_SOURCE_LENGTH = 256
MAX_TARGET_LENGTH = 256

def preprocess_function(examples):
    inputs = examples["transliteration"]
    targets = examples["translation"]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_SOURCE_LENGTH,
        truncation=True,
        padding=False
    )

    labels = tokenizer(
        text_target=targets,
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding=False
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH)

Map:   0%|          | 0/1404 [00:00<?, ? examples/s]

Map:   0%|          | 0/157 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

In [4]:
import math
from collections import Counter
import numpy as np

# Pure-Python BLEU / chrF

def _ngram_counts(tokens, n):
    return Counter(tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1))

def corpus_bleu(preds, refs, max_n=4):
    clipped_counts = [0] * max_n
    total_counts = [0] * max_n
    pred_len_total, ref_len_total = 0, 0
    for pred, ref in zip(preds, refs):
        p_tokens, r_tokens = pred.split(), ref.split()
        pred_len_total += len(p_tokens)
        ref_len_total += len(r_tokens)
        for n in range(1, max_n + 1):
            p_counts = _ngram_counts(p_tokens, n)
            r_counts = _ngram_counts(r_tokens, n)
            clipped_counts[n - 1] += sum(min(c, r_counts[g]) for g, c in p_counts.items())
            total_counts[n - 1] += max(sum(p_counts.values()), 0)
    precisions = [clipped_counts[n] / total_counts[n] if total_counts[n] > 0 else 0.0 for n in range(max_n)]
    geo_mean = 0.0 if min(precisions) == 0 else math.exp(sum(math.log(p) for p in precisions) / max_n)
    bp = 1.0 if pred_len_total > ref_len_total else math.exp(1 - ref_len_total / max(pred_len_total, 1))
    return 100.0 * bp * geo_mean

def corpus_chrf(preds, refs, n=6, beta=2):
    tp = [0] * n; tt = [0] * n; rc = [0] * n; rt = [0] * n
    for pred, ref in zip(preds, refs):
        p_chars, r_chars = pred.replace(" ", ""), ref.replace(" ", "")
        for k in range(1, n + 1):
            p_counts, r_counts = _ngram_counts(p_chars, k), _ngram_counts(r_chars, k)
            overlap = sum((p_counts & r_counts).values())
            tp[k - 1] += overlap; tt[k - 1] += max(sum(p_counts.values()), 0)
            rc[k - 1] += overlap; rt[k - 1] += max(sum(r_counts.values()), 0)
    precisions = [tp[k] / tt[k] if tt[k] > 0 else 0.0 for k in range(n)]
    recalls = [rc[k] / rt[k] if rt[k] > 0 else 0.0 for k in range(n)]
    avg_p, avg_r = sum(precisions) / n, sum(recalls) / n
    if avg_p + avg_r == 0:
        return 0.0
    beta2 = beta ** 2
    return 100.0 * (1 + beta2) * avg_p * avg_r / (beta2 * avg_p + avg_r)

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    # Replace -100 in labels so they decode correctly
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [label.strip() for label in decoded_labels]

    chrf_pp = corpus_chrf(decoded_preds, decoded_labels)  # beta=2 approximates chrF++
    bleu = corpus_bleu(decoded_preds, decoded_labels)

    return {
        "chrf_pp": round(chrf_pp, 4),
        "bleu": round(bleu, 4)
    }


In [5]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

RUN_NAME = "mbart50-clean-finetuning"

set_seed(42)

training_args = Seq2SeqTrainingArguments(
    output_dir="./results_mbart",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=4,        # Increased if VRAM allows
    per_device_eval_batch_size=16,       # Increased from 1 to speed up eval
    gradient_accumulation_steps=2,
    gradient_checkpointing=True,
    optim="adafactor",
    generation_max_length=MAX_TARGET_LENGTH,
    generation_num_beams=1,
    weight_decay=0.01,
    save_total_limit=1,
    num_train_epochs=5,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    logging_steps=10,                    # Log more frequently
    load_best_model_at_end=True,
    metric_for_best_model="chrf_pp",
    greater_is_better=True,
    report_to="none",
    disable_tqdm=False                   # Ensures progress logs fallback to standard output
)

model.gradient_checkpointing_enable()

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Chrf Pp,Bleu
1,7.094638,1.707731,38.087600,14.158700
2,5.624204,1.503684,43.193600,16.942700
3,4.349630,1.427763,45.436900,20.855600
4,3.897773,1.418440,47.048500,21.544900
5,3.323656,1.432435,48.175300,22.342200


The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=440, training_loss=5.230035781860352, metrics={'train_runtime': 2303.5124, 'train_samples_per_second': 3.048, 'train_steps_per_second': 0.191, 'total_flos': 3801196405063680.0, 'train_loss': 5.230035781860352, 'epoch': 5.0})

In [6]:
# Save the trained model and tokenizer locally instead of pushing to the Hugging Face Hub
# (no internet available). These land in Kaggle's working directory and can be kept as
# notebook output / attached to a new dataset version if you want to reuse the weights.
model.save_pretrained("./subword-seq2seq-model")
tokenizer.save_pretrained("./subword-seq2seq-tokenizer")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./subword-seq2seq-tokenizer/tokenizer_config.json',
 './subword-seq2seq-tokenizer/tokenizer.json')

In [7]:
# Generate predictions on the validation dataset
raw_predictions = trainer.predict(tokenized_datasets["validation"])
preds = np.where(raw_predictions.predictions != -100, raw_predictions.predictions, tokenizer.pad_token_id)
decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

val_df_results = pd.DataFrame({
    "id": val_df["oare_id"].values,
    "source_text": val_df["transliteration"].values,
    "target_text": val_df["translation"].values,
    "pred_translation": [p.strip() for p in decoded_preds]
})

val_df_results.to_csv("preds_val_mbart50.csv", index=False)
print("Predictions file preds_val_mbart50.csv saved successfully!")


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Predictions file preds_val_mbart50.csv saved successfully!


In [8]:
import os
import pandas as pd
import torch
from tqdm import tqdm

test_path = "/kaggle/input/competitions/deep-past-initiative-machine-translation/test.csv"

test_df = pd.read_csv(test_path)

model.eval()
model.to("cuda")

# Ensure tokenizer pad token is set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Get forced BOS token ID to enforce English output in mBART
forced_bos_token_id = tokenizer.lang_code_to_id.get("en_XX")

predictions = []

# Generate outputs row-by-row
for text in tqdm(test_df["transliteration"], desc="Generating Submissions"):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_TARGET_LENGTH,
            pad_token_id=tokenizer.pad_token_id,
            forced_bos_token_id=forced_bos_token_id
        )

    # Encoder-decoder models output target tokens directly; decode outputs[0] without slicing
    decoded_output = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
    predictions.append(decoded_output)

# Form output DataFrame
submission = pd.DataFrame({
    "id": test_df["id"],
    "translation": predictions
})

# Export to root directory without DataFrame index
submission.to_csv("submission.csv", index=False)
print("Saved submission.csv successfully!")


Generating Submissions: 100%|██████████| 4/4 [00:04<00:00,  1.12s/it]

Saved submission.csv successfully!
